# Is Dar es Salaam warmer than forty years ago?

**MATH4SDG / UDSM — optional project, not a compulsory lesson**

**Research question:** Has the *NASA POWER estimated* annual mean two-metre air temperature of the grid cell containing Dar es Salaam increased over 1986–2025? How confident should we be?

**Crucial distinction:** NASA POWER T2M is a **gridded meteorological reanalysis estimate** (primarily based on MERRA-2), not a direct thermometer record from Dar es Salaam Airport. It can have excellent temporal coverage while still having model biases, spatial averaging and changes in assimilated observations. See [official methodology](https://power.larc.nasa.gov/docs/methodology/data/sources_meteorology/).

**Source selection:** The notebook uses a locally downloaded NASA POWER CSV **if present**. Otherwise it uses an explicitly labeled **SYNTHETIC** dataset only for code practice. Synthetic output must NEVER be reported as evidence about real climate.

**Before the bootcamp, while online**, download using:

```sh
uv run python tools/download_nasa_power.py --location dar_es_salaam
```

The private CSV and provenance metadata are excluded from Git. See [project README](../projects/nasa-power-temperature/README.md) for details.

This activity adapts ideas from the [MAT102 Meteostat notebooks](https://github.com/mbr085/V26MAT102) but gives data-source quality, statistical uncertainty and interpretation equal importance to fitting a model. Code: MIT; teaching prose: CC BY 4.0; datasets: separate terms.

**Python quick references:** [pandas cheat sheet](../reference/pandas_cheatsheet.ipynb) · [Matplotlib cheat sheet](../reference/matplotlib_cheatsheet.ipynb). Use these when a library operation is unfamiliar; the mathematical investigation remains the focus.

**Working style:** This notebook uses short, visible pandas and NumPy calculations, not hidden analysis functions. You can use the linked cheat-sheet notebooks as references, and you are encouraged to check each step independently.


## 1. Make the scientific question precise

A single hot day says little about long-term temperature change. We will estimate a **descriptive linear slope** in °C/decade from annual temperature summaries, then compare the first and last ten years.

We are studying the POWER **grid-cell estimate**, not direct temperature measurements at a specific airport, nor the whole country. What additional evidence would justify a statement about Dar es Salaam residents' experienced climate, or about East Africa?


## 2. Load the temperature series

We will use the **locally downloaded NASA POWER CSV** if it exists. Otherwise we generate **clearly labelled synthetic data** so you can practice offline.

Start JupyterLab from the project root and keep this notebook inside `notebooks/`. The small `bootstrap` helper provides the project directory; you do not have to manage Python import paths.


In [ ]:
import numpy as np
import pandas as pd

import bootstrap
from tools.generate_synthetic_temperature import generate

power_csv = (
    bootstrap.PROJECT_ROOT
    / "data/private/nasa_power_dar_es_salaam_1986_2025.csv"
)


First, determine **which source we are using**. All subsequent conclusions must keep this distinction clear.


In [ ]:
if power_csv.exists():
    data = pd.read_csv(power_csv, parse_dates=["date"])
    source = "NASA POWER T2M — GRIDDED REANALYSIS ESTIMATES (NOT STATION OBSERVATIONS)"
else:
    data = generate()
    source = "SYNTHETIC DATA ONLY — NOT EVIDENCE ABOUT CLIMATE"

print(source)


If you downloaded NASA POWER data, check the accompanying provenance file. Missing provenance information is a reason to investigate the dataset, not a reason to call the series observed station temperatures.


In [ ]:
if power_csv.exists():
    metadata_file = power_csv.with_suffix(".json")
    print("Provenance JSON:", metadata_file if metadata_file.exists() else "MISSING — re-download recommended")
else:
    print("No external data or provenance file needed: synthetic practice only.")


Prepare the date index and ensure the requested period is present. We **reject repeated dates** and convert invalid temperature entries to missing values rather than inventing temperatures.


In [ ]:
data = data.sort_values("date").copy()
if data["date"].duplicated().any():
    raise ValueError("Duplicate dates: verify the source before proceeding.")

data["temp"] = pd.to_numeric(data["temp"], errors="coerce")
data = data.set_index("date").loc["1986":"2025"]
if data.empty:
    raise ValueError("No data in the requested 1986–2025 period.")


Inspect the first rows. The table contains daily values, but these values are estimates when `source` identifies NASA POWER.


In [ ]:
data.head()


## 3. Is the time series complete?

Before fitting a line, check missing calendar days. A grid-cell reanalysis series can cover all dates while still having **systematic biases and model-dependent estimates**. Complete **coverage is not independent observational validation**.

Missing temperatures must not be silently forward-filled.


Convert the series to an explicit daily calendar. Missing dates become missing values (`NaN`).


In [ ]:
daily = data["temp"].asfreq("D")
daily.head()


Count how many calendar days have no value.


In [ ]:
pd.Series({
    "Calendar days": len(daily),
    "Days with temperature values": daily.notna().sum(),
    "Missing days": daily.isna().sum(),
})


Calculate the fraction of days with a value in each year. Here `groupby()` groups Boolean indicators by calendar year, and the mean is the fraction that are `True`.


In [ ]:
coverage = daily.notna().groupby(daily.index.year).mean()
coverage.nsmallest(8).round(3)


Inspect the beginning of the series. Leave gaps visible instead of drawing a made-up continuous curve.


In [ ]:
ax = daily.iloc[:730].plot(
    figsize=(10, 3),
    title="First two years — NASA POWER grid cell or synthetic example",
)
ax.set_xlabel("Date")
ax.set_ylabel("Temperature (°C)");


Plot year-by-year coverage separately. If the curve is near 1, the series has few missing entries, **not necessarily few scientific uncertainties**.


In [ ]:
ax = coverage.plot(
    marker="o", figsize=(9, 3),
    title="Fraction of calendar days with a temperature value",
)
ax.set_xlabel("Year")
ax.set_ylabel("Coverage fraction")
ax.set_ylim(0, 1.05);


## 4. Seasonal variation

Plot the mean temperature by calendar month. Seasonal variation may dominate the short-term record. Why is a comparison of the same months in different years more meaningful than a comparison of individual days?

Reanalysis temperatures represent a spatial grid cell. Even the shape of the seasonal cycle should ideally be compared with trusted observations.


In [ ]:
seasonal = daily.groupby(daily.index.month).mean()
seasonal.round(2)


The monthly pattern is a descriptive summary. Months with substantially different amounts of missing data may have different reliability.


In [ ]:
ax = seasonal.plot(
    marker="o", figsize=(8, 3),
    title="Average temperature by calendar month",
)
ax.set_xlabel("Calendar month")
ax.set_ylabel("Temperature (°C)");


## 5. Construct annual temperatures from daily values

This mathematical calculation should be visible, because **how we summarize the daily data affects the trend**.

For each calendar month:

1. Calculate the mean of available daily temperatures.
2. Require **at least 20 valid daily values**; otherwise reject that month's mean.
3. For each year, take the *unweighted average of its 12 monthly means*, **only if all 12 months qualify**.

The equal-weight monthly convention is deliberate: it gives the same weight to each calendar month rather than giving longer months greater weight. It is not the only scientifically possible annual summary.

We do **not** fill missing months with nearby values. The [pandas cheat-sheet notebook](../reference/pandas_cheatsheet.ipynb) explains `resample()` and missing dates.


**Step 1 — Calculate monthly counts and means.** `resample("MS")` groups daily data into calendar months, labelled by the **month start**.


In [ ]:
monthly = pd.DataFrame({
    "valid_days": daily.resample("MS").count(),
    "mean_temp": daily.resample("MS").mean(),
})
monthly.head()


**Step 2 — Keep only monthly means based on at least 20 valid days.** Other months receive `NaN` in the new column.


In [ ]:
monthly["eligible_mean"] = monthly["mean_temp"].where(
    monthly["valid_days"] >= 20
)
monthly.loc["1997-01":"1997-07"]


**Step 3 — Count eligible months and average their monthly means for each year.** `resample("YS")` groups by calendar year and labels the result by the **year start**.


In [ ]:
months_per_year = monthly["eligible_mean"].resample("YS").count()
annual_candidates = monthly["eligible_mean"].resample("YS").mean()

pd.DataFrame({
    "Eligible months": months_per_year,
    "Mean of eligible months": annual_candidates,
}).head()


**Step 4 — Keep only years with all 12 eligible months.** Display the number of eligible years and the excluded years explicitly.


In [ ]:
annual = annual_candidates.loc[months_per_year == 12].copy()
annual.index = annual.index.year
annual.index.name = "year"
annual.name = "mean_temp"

pd.Series({
    "Eligible years": len(annual),
    "Calendar years checked": len(months_per_year),
})


In [ ]:
excluded_years = months_per_year.index[months_per_year < 12].year.tolist()
excluded_years


**Quality safeguard:** a regression based on very few complete annual summaries would be misleading for our 40-year question.


In [ ]:
if len(annual) < 15:
    raise ValueError("Fewer than 15 eligible years. Investigate missingness before fitting.")


**Independent check:** the original reference implementation remains available in `tools/temperature_methods.py`. We compare the student-visible steps with it, rather than using that helper as a black box to perform the calculation.


In [ ]:
from tools.temperature_methods import annual_means

reference_annual, _ = annual_means(daily, min_days_per_month=20)
pd.testing.assert_series_equal(annual, reference_annual)
print("Annual summaries match the independent reference implementation.")


Now inspect the eligible annual values themselves.


In [ ]:
annual.head()


In [ ]:
ax = annual.plot(
    marker="o", figsize=(10, 4),
    title="Annual mean — POWER grid cell or synthetic example",
)
ax.set_xlabel("Year")
ax.set_ylabel("Annual temperature (°C)");


## 6. Fit a straight line with least squares

We model annual mean temperature by

$$T_y = \alpha + \beta(y-y_0) + \varepsilon_y.$$

The slope `β` is measured in °C **per year**. Multiplying by 10 gives °C **per decade**.

Ordinary least squares chooses the coefficients to minimize
$$\sum_i (T_i-\alpha-\beta t_i)^2.$$

We will calculate the coefficients directly using NumPy rather than hiding them inside a helper function. Does a line adequately describe the annual pattern?


First prepare the arrays, using the first eligible year as the time origin.


In [ ]:
years = annual.index.to_numpy(dtype=float)
temperatures = annual.to_numpy(dtype=float)
time = years - years.min()

time[:5]


Build the **design matrix**: the first column consists of ones (the intercept); the second contains elapsed years. [NumPy least squares](https://numpy.org/doc/stable/reference/generated/numpy.linalg.lstsq.html) solves this linear problem.


In [ ]:
design = np.column_stack([np.ones(len(time)), time])
design[:5]


Fit the coefficients and calculate the fitted annual values. The expression `design @ coefficients` is matrix-vector multiplication.


In [ ]:
coefficients = np.linalg.lstsq(design, temperatures, rcond=None)[0]
intercept, slope_per_year = coefficients
fitted = design @ coefficients

slope_per_decade = 10 * slope_per_year
round(float(slope_per_decade), 3)


Plot the annual summaries and fitted line together. **The line is a model summary, not an additional temperature observation.**


In [ ]:
ax = annual.plot(
    marker="o", linestyle="none", figsize=(10, 4),
    label="Annual summaries",
    title="POWER grid-cell trend (or labelled synthetic example)",
)
ax.plot(years, fitted, label="Least-squares line")
ax.set_xlabel("Year")
ax.set_ylabel("Annual temperature (°C)")
ax.legend();


## 7. Statistical intervals versus scientific confidence

The fitted slope is an estimate, not a known constant. We compare:

- A **conventional OLS 95% confidence interval**, relying on a simple regression error model.
- A **HAC (heteroskedasticity/autocorrelation-consistent) interval** allowing some heteroskedasticity and short-range serial dependence, with **three annual lags**.

Both intervals are conditional on assumptions about the annual series. Neither captures uncertainty from the **reanalysis input and model, spatial smoothing, or changes in observing systems**. A 95% confidence interval is not a 95% posterior probability that a fixed slope parameter lies in the interval.

What evidence could undermine a narrow interval even if the software reports it accurately?


Use SciPy as an **independent check of the least-squares slope**. It also provides a standard error under the conventional model.


In [ ]:
from scipy import stats

check = stats.linregress(years, temperatures)
assert np.isclose(check.slope, slope_per_year, atol=1e-10)
round(float(10 * check.slope), 3)


The conventional 95% interval uses the Student t critical value with `n−2` degrees of freedom.


In [ ]:
tcrit = stats.t.ppf(0.975, df=len(years) - 2)
conventional_ci = 10 * np.array([
    slope_per_year - tcrit * check.stderr,
    slope_per_year + tcrit * check.stderr,
])
conventional_ci.round(3)


For the HAC interval, fit **the same linear regression** with statsmodels, changing how the coefficient uncertainty is estimated. The `maxlags=3` choice is a sensitivity assumption, not a correction for all possible climate-data problems.


In [ ]:
import statsmodels.api as sm

hac_fit = sm.OLS(temperatures, design).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 3},
)
hac_ci = 10 * np.asarray(hac_fit.conf_int(alpha=0.05))[1]
hac_ci.round(3)


Compare the intervals on the same scale: °C per decade. A smaller interval is not automatically more trustworthy.


In [ ]:
intervals = pd.DataFrame({
    "Method": ["Conventional OLS", "HAC (3 annual lags)"],
    "Slope (°C/decade)": [slope_per_decade, slope_per_decade],
    "95% lower": [conventional_ci[0], hac_ci[0]],
    "95% upper": [conventional_ci[1], hac_ci[1]],
})
intervals.round(3)


Inspect the residuals and their one-year lag correlation. A small correlation does **not** certify that all error-model assumptions hold.


In [ ]:
residuals = temperatures - fitted
pd.Series(residuals).autocorr(lag=1)


In [ ]:
ax = pd.Series(residuals, index=annual.index).plot(
    marker="o", figsize=(10, 3),
    title="Residuals from the full-period least-squares line",
)
ax.axhline(0, linestyle="--", color="gray")
ax.set_xlabel("Year")
ax.set_ylabel("Residual (°C)");


## 8. Compare the earliest and latest decades

If at least eight adequately covered years are present in each decade, compare **1986–1995** with **2016–2025**. This answers a different question from the slope of a fitted straight line. If the data do not meet the minimum, we should report that rather than impute years.

Even a convincing difference concerns the **reanalysis grid-cell series**; separate corroboration is needed for an observational or regional warming claim.


Compare the first and last ten-year windows without substituting missing annual values. Report how many of each decade's years qualify.


In [ ]:
early = annual.loc[1986:1995]
late = annual.loc[2016:2025]

pd.Series({
    "Eligible years, 1986–1995": len(early),
    "Eligible years, 2016–2025": len(late),
})


We require at least **8 eligible years in each decade** to make this descriptive comparison. This is not the same estimand as the fitted slope.


In [ ]:
if len(early) >= 8 and len(late) >= 8:
    decade_difference = late.mean() - early.mean()
    print(f"2016–2025 minus 1986–1995: {decade_difference:.3f} °C")
else:
    print("Insufficient eligible years for the decade comparison.")


## 9. Does the estimated warming depend on the starting year?

**Our central follow-up question:** How much confidence should we place in the estimated trend if the answer changes when we choose a different starting year?

Fit separate lines to annual temperature summaries for **1986–2025, 1991–2025, 1996–2025, and 2001–2025**. Each fit uses a different span of the *same* NASA POWER reanalysis series. Calculate the slope in °C per decade, an interval allowing for possible annual autocorrelation (HAC, with three annual lags), and the number of eligible years.

- Do you see a systematic change in the estimated slopes?
- Can a 40-year positive slope coexist with a small slope over the final 25 years?
- Does a small slope mean that warming has stopped? What does its confidence interval say?
- The windows **overlap**. Why is comparing two individual confidence intervals *not* a valid significance test for their difference?
- What would count as evidence that the trend itself has changed, rather than that a shorter fit is more sensitive to variability?

Do not pick the start year that gives the answer you prefer. Any inference about climate is conditional on the reanalysis data and statistical assumptions; this is **not** direct evidence about a particular weather station.


Make one fitted line for each starting year while keeping **2025 as the common end year**. We show the regression and HAC computation explicitly inside the loop, rather than calling a hidden fitting helper.

A short window can produce a different slope simply because it includes different temperatures. Four fitted slopes from overlapping windows are **not independent**.


In [ ]:
sensitivity = []

for first_year in [1986, 1991, 1996, 2001]:
    part = annual.loc[first_year:2025]
    if len(part) < 15:
        continue

    part_years = part.index.to_numpy(dtype=float)
    part_time = part_years - part_years.min()
    part_design = np.column_stack([np.ones(len(part_time)), part_time])

    part_fit = sm.OLS(part.to_numpy(dtype=float), part_design).fit(
        cov_type="HAC", cov_kwds={"maxlags": 3}
    )
    lower, upper = 10 * np.asarray(part_fit.conf_int(alpha=0.05))[1]

    sensitivity.append({
        "start_year": first_year,
        "eligible_years": len(part),
        "slope_C_per_decade": 10 * part_fit.params[1],
        "HAC_95_lower": lower,
        "HAC_95_upper": upper,
    })

sensitivity_table = pd.DataFrame(sensitivity)
if sensitivity_table.empty:
    raise ValueError("Not enough eligible years for the sensitivity analysis.")


See the numerical results first. **Do not choose the starting year that gives your preferred conclusion.**


In [ ]:
sensitivity_table.round(3)


Now plot those slopes with their individual HAC confidence intervals. These intervals are **not** a test of whether two overlapping-window slopes differ.


In [ ]:
import matplotlib.pyplot as plt

x = sensitivity_table["start_year"].to_numpy()
y = sensitivity_table["slope_C_per_decade"].to_numpy()
lower = sensitivity_table["HAC_95_lower"].to_numpy()
upper = sensitivity_table["HAC_95_upper"].to_numpy()

fig, ax = plt.subplots(figsize=(9, 4))
ax.errorbar(x, y, yerr=[y - lower, upper - y], fmt="o", capsize=4)
ax.axhline(0, linestyle="--", color="gray")
ax.set_xticks(x)
ax.set_xlabel("Starting year (all periods end in 2025)")
ax.set_ylabel("Fitted slope (°C/decade)")
ax.set_title("Nested periods with HAC 95% intervals");


### Optional investigation: is there evidence of a change in slope?

Comparing the individual intervals is not a valid test of whether the *slopes differ*, because the data windows overlap. An advanced group could explore a **single continuous segmented-regression model** with a candidate change point at 2001:

$$
T_y = \alpha + \beta (y-1986) + \gamma\max(0,y-2001) + \varepsilon_y.
$$

Here the model's slope is $\beta$ before 2001 and $\beta+\gamma$ after 2001. Investigate the hypothesis $\gamma=0$ with a confidence interval that accounts for temporal dependence.

**Important:** This tests a different model and question from comparing the two nested least-squares slopes above. Choosing 2001 because it gave an interesting result and then treating it as a pre-specified change point would overstate the evidence. A convincing investigation should consider sensitivity to the choice of change point, nonlinearity, and uncertainty about the reanalysis source.

**Other challenges:** Re-run annual aggregation with 15 or 25 valid days per month; ask an AI assistant to critique three assumptions, then verify its criticisms; inspect [NASA POWER methodology](https://power.larc.nasa.gov/docs/methodology/data/sources_meteorology/) or the optional [Meteostat missingness exercise](meteostat_station_missingness.ipynb) to discuss data provenance.

**Group report:** State the precise estimand, full-period estimate and interval, starting-year sensitivity, what can and cannot be inferred about a change in trend, source limitations, and a proposed next step. It is acceptable to conclude that the data do not justify a particular claim. **Never present synthetic results as actual observations.**


### Extensions

- Compare the POWER grid cells for Dar es Salaam and Dodoma, using separate downloads and accounting for spatial smoothing.
- Fit monthly temperature anomalies or seasonal sine/cosine regression, then compare with the annual analysis.
- Consider climate mechanisms and independent observations, but do **not** infer causal attribution from a single fitted trend.
- Use the optional Meteostat station record to study why a time series can be unsuitable even when the regression routine happily returns a slope.

Sources: [NASA POWER daily API](https://power.larc.nasa.gov/docs/services/api/temporal/daily/), [data sources](https://power.larc.nasa.gov/docs/methodology/data/sources_meteorology/), [referencing guide](https://power.larc.nasa.gov/docs/referencing/).

**Using AI critically:** Ask it to explain a result or suggest an independent check, then verify the calculation and the assumptions. Do not send private industry data or restricted records to external services. In a group report, always make clear whether the analysis used the NASA POWER grid-cell CSV or the offline synthetic substitute.
